# Plan v4 on TIDE (M1–M4 + Track B/C architecture compare)

**Spawn:** PyTorch Notebook, 1× L40, 8 CPU, 32 GB RAM. Plan of record: `agent_plan.md`. NCUGD is no longer evaluated.

**Track A** = YOLO-seg (E1/E2/E4…). **Track B** = U-Net (`train_semseg`, E3). **Track C** = RF-DETR-Seg Large (`train_rfdetr`, E7). Do **not** launch E3 with `train_yolo` or E7 with `train_yolo`.

## Status snapshot (local `outputs/` as of 2026-11-17)

| Step | Status | Gate note (recall @ P≥0.85 on merged test) |
|---|---|---|
| M1 R0–R2 | done | R2_1024 **0.847** best glass; R1 best AGDD `images/` crack |
| E4 (3 seeds) | done | **0.549 ± 0.278** — does **not** beat E1 **0.825 ± 0.010** |
| E4b (3 seeds) | done | **0.259 ± 0.075** — fails gate; lower FP/100 |
| E3 | s0+s2 eval’d; **s1 needs `run_eval`** | threshold-free @P≥0.85 is 0 (soft scores); AGDD strong |
| E5pre / E6pre | train done | proxy only (`eval: false`); glass FT (E5/E6) not started |
| E7 RF-DETR | code ready on `models` | first: Large smoke on TIDE, then `E7_s0` |

Order remaining (one GPU job at a time):
1. Pull + setup (`models` branch has Track C)
2. Finish **E3_s1 eval** if missing; optional architecture table
3. **Track C** smoke → `E7_s0` (then seeds 1–2)
4. **M4 FT**: `E5` from `E5pre` (then `E6` only if E5 helps) — E4/E4b already failed the YOLO stack gate
5. Threshold-free compare: YOLO vs U-Net vs RF-DETR

## 1. Pull and setup

In [ ]:
%%bash
cd ~/glass_crack/cracked && git fetch --all && git checkout models && git pull --ff-only && git log -1 --oneline
# export EXTRAS_URL="https://drive.google.com/file/d/<id>/view"   # only if extras.zip is not uploaded
bash ~/glass_crack/cracked/scripts/tide/setup.sh 2>&1 | tail -30
# Confirm Track C pin landed (rfdetr[train]==1.12.1)
"$HOME/glass_crack/venv/bin/python" -c "import rfdetr; print('rfdetr', getattr(rfdetr,'__version__', '?'))" 2>/dev/null || echo "rfdetr missing — re-run setup.sh"
ls ~/glass_crack/bundle/extras 2>/dev/null

## 2. One-time: archive the old E4 (scale/colour only) as E4old
The E4 config now also has motion blur + JPEG; without this the new E4_s0 would report "already finished".

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh && cd "$CODE_DIR"
for d in runs evals; do
  [ -d "$OUT_ROOT/$d/E4_s0" ] && [ ! -d "$OUT_ROOT/$d/E4old_s0" ] && mv "$OUT_ROOT/$d/E4_s0" "$OUT_ROOT/$d/E4old_s0" && echo "moved $d/E4_s0 -> E4old_s0"
done
[ -f "$OUT_ROOT/logs/E4_s0.log" ] && mv "$OUT_ROOT/logs/E4_s0.log" "$OUT_ROOT/logs/E4old_s0.log" && rm -f "$OUT_ROOT/logs/E4_s0.pid"
sed -i 's/"exp_id": "E4"/"exp_id": "E4old"/' "$OUT_ROOT"/evals/E4old_s0/*.json  # compare_runs groups by this
sed -i 's/^E4,/E4old,/' "$OUT_ROOT/results/results.csv" && grep -c '^E4old,' "$OUT_ROOT/results/results.csv"

## 3. M1: resolution ablation on frozen E1_s0 (GPU, ~5-15 min per variant)
Same checkpoint, threshold re-frozen on valid per variant. R0 whole @640, R1 640 px tiles over the image at long side 1280, R2 whole @1024 / @1280.

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh && cd "$CODE_DIR"
CK="$OUT_ROOT/runs/E1_s0/weights/best.pt"
run() { "$PY" -m glass_crack_models.eval.run_eval --checkpoint "$CK" --seed 0 --device 0 --no-map \
          --task yolo26s-seg --notes "M1 $*" --compute-units "TIDE $GPU_NAME" "$@" 2>&1 | tail -4; }
run --exp-id R0 --imgsz 640 --whole
run --exp-id R1 --imgsz 1280
run --exp-id R2_1024 --imgsz 1024 --whole
run --exp-id R2_1280 --imgsz 1280 --whole
"$PY" -m glass_crack_models.eval.compare_runs --runs E1_s0 R0_s0 R1_s0 R2_1024_s0 R2_1280_s0 --out "$OUT_ROOT/results/m1"
cat "$OUT_ROOT/results/m1/compare_runs.md"

## 4–6. Training / eval jobs (background; survive closed tabs, not a server stop)
Set `CFG` / `NAME` / `SEED` / `MODULE`, run, then monitor below. Rerunning the same command resumes.

| Step | CFG | NAME | MODULE | Seeds | Notes |
|---|---|---|---|---|---|
| Track B | `e3_unet_r34.yaml` | `E3` | `train_semseg` | 0–2 | s0+s2 done; **eval E3_s1** if missing |
| M2 | `e4_yolo_robust.yaml` | `E4` | `train_yolo` | 0–2 | done — failed E1 gate |
| M3 | `e4b_yolo_hardneg.yaml` | `E4b` | `train_yolo` | 0–2 | done — failed E1 gate |
| M4 A | `e5_pre_cubit.yaml` → `e5_yolo_cubit_ft.yaml` | `E5pre` / `E5` | `train_yolo` | pre:0; FT:0–2 | pretrain done; FT next |
| M4 B | `e6_pre_cardd.yaml` → `e6_yolo_cardd_ft.yaml` | `E6pre` / `E6` | `train_yolo` | pre:0; FT:0–2 | pretrain done; only if E5 helps |
| **Track C** | `e7_rfdetr_seg.yaml` | `E7` | `train_rfdetr` | 0 (then 1–2) | Large, batch 2 × accum 8 |

U-Net / RF-DETR use `--device cuda`. YOLO uses `--device 0`. E8 (audited labels) waits for the manual crack/scratch audit.

**E3_s1 eval only** (weights exist, no `outputs/evals/E3_s1`):
```bash
"$PY" -m glass_crack_models.eval.run_eval \
  --checkpoint "$OUT_ROOT/runs/E3_s1/best.pth" --exp-id E3 --seed 1 \
  --device cuda --task unet-resnet34 --compute-units "TIDE $GPU_NAME"
```

**Track C first time on TIDE** — 1-epoch smoke before full E7:
```bash
bash "$CODE_DIR/scripts/tide/job.sh" start E7_s0_smoke glass_crack_models.train.train_rfdetr \
  --config e7_rfdetr_seg.yaml --seed 0 --smoke --device cuda --workers "$WORKERS" \
  --compute-units "TIDE $GPU_NAME"
```

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh
# Pick ONE block. Rerunning resumes.

# Track C RF-DETR-Seg Large (architecture compare) — preferred next train:
CFG=e7_rfdetr_seg.yaml; NAME=E7; SEED=0; MODULE=glass_crack_models.train.train_rfdetr; DEV=cuda

# M4 A fine-tune after E5pre (weights: $OUT_ROOT/runs/E5pre_s0/weights/best.pt):
# CFG=e5_yolo_cubit_ft.yaml; NAME=E5; SEED=0; MODULE=glass_crack_models.train.train_yolo; DEV=0

# U-Net (Track B) — do NOT use train_yolo:
# CFG=e3_unet_r34.yaml; NAME=E3; SEED=1; MODULE=glass_crack_models.train.train_semseg; DEV=cuda

# YOLO (Track A) legacy recipes (E4/E4b already scored; failed E1 gate):
# CFG=e4_yolo_robust.yaml; NAME=E4; SEED=0; MODULE=glass_crack_models.train.train_yolo; DEV=0

bash "$CODE_DIR/scripts/tide/job.sh" start ${NAME}_s$SEED $MODULE \
    --config $CFG --seed $SEED --device $DEV --workers "$WORKERS" --compute-units "TIDE $GPU_NAME"
bash "$CODE_DIR/scripts/tide/job.sh" status

In [ ]:
%%bash
# Monitor / stop. Change the job name as needed.
J=~/glass_crack/cracked/scripts/tide/job.sh
bash "$J" status
echo
bash "$J" tail E7_s0 30
# bash "$J" tail E5_s0 30
# bash "$J" stop E7_s0

## 7. Threshold-free comparison and results table (CPU)
Headline gate for YOLO stacking remains E1. Architecture check: YOLO (E1) vs U-Net (E3) vs RF-DETR (E7). E4/E4b are included in the full table for the record even though they failed the gate.

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh && cd "$CODE_DIR"
"$PY" -m glass_crack_models.eval.compare_runs
cat "$OUT_ROOT/results/compare_runs.md"
echo

# Architecture trio (skips missing dirs). Prefer full 3-seed E3 + E7 when present.
ARCH_RUNS="E1_s0 E1_s1 E1_s2"
for r in E3_s0 E3_s1 E3_s2 E7_s0 E7_s1 E7_s2; do
  [ -d "$OUT_ROOT/evals/$r" ] && ARCH_RUNS="$ARCH_RUNS $r"
done
echo "=== Architecture compare: $ARCH_RUNS ==="
"$PY" -m glass_crack_models.eval.compare_runs --runs $ARCH_RUNS --out "$OUT_ROOT/results/yolo_unet_rfdetr"
cat "$OUT_ROOT/results/yolo_unet_rfdetr/compare_runs.md"

echo
echo "=== YOLO stack (E1 / E4 / E4b) ==="
"$PY" -m glass_crack_models.eval.compare_runs \
  --runs E1_s0 E1_s1 E1_s2 E4_s0 E4_s1 E4_s2 E4b_s0 E4b_s1 E4b_s2 \
  --out "$OUT_ROOT/results/yolo_stack"
cat "$OUT_ROOT/results/yolo_stack/compare_runs.md"

In [ ]:
import os
from pathlib import Path
import pandas as pd

root = Path(os.path.expanduser("~/glass_crack/outputs"))
df = pd.read_csv(root / "results/results.csv").drop_duplicates(
    ["exp_id", "seed", "checkpoint_sha256"], keep="last"
)
cols = ["exp_id", "track", "task", "seed", "threshold", "test_recall", "test_precision",
        "test_fp100", "test_mask_iou", "agdd_images_recall", "agdd_images_crack_recall",
        "agdd_image_recall", "agdd_image_crack_recall", "notes"]
display(df[[c for c in cols if c in df.columns]].sort_values(["exp_id", "seed"]))

# Quick presence check for unfinished / next runs
need = ["E3_s1", "E5_s0", "E6_s0", "E7_s0", "E7_s0_smoke"]
rows = []
for name in need:
    run = root / "runs" / name
    ev = root / "evals" / name
    rows.append({"run": name, "weights": run.exists(), "eval": ev.exists()})
display(pd.DataFrame(rows))

md = root / "results/compare_runs.md"
if md.exists():
    print(md.read_text())